# Lesson 13 · From empirical Bayes to full Bayes with PyMC

**Source:** Beyond the book. It picks up the double-dipping problem from Chapter 8.4 and the small-sample failures from Chapter 13, and bridges to Statistical Rethinking (Chapters 12–13) and Cardprice notebook 07.

**Question:** What changes when we stop plugging in a fitted prior and let the model carry its uncertainty about the prior too? When does it matter?

**Goal:** Fit the beta-binomial hierarchy in PyMC, compare it with empirical Bayes on 10,000 players (where they should agree) and on 30 (where they shouldn't), and get comfortable with the PyMC 6 / ArviZ 1 API.

**Contents**

1. Setup and the PyMC 6 / ArviZ 1 API
2. The hierarchical beta-binomial model
3. Fit it to all players
4. Compare with empirical Bayes
5. Thirty players: where full Bayes earns its keep
6. Regression in the hierarchy (optional)
7. Card connection: simulated pull rates

**Tags:** `full-bayes` `pymc` `hierarchical-models` `beta-binomial` `mcmc` `arviz` `partial-pooling` `beyond-the-book`

**Before you start:** Lessons 02, 06 and 12, and some PyMC. Start Jupyter from the activated `pymc` environment.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
# (If ENVIRONMENT_NOTES.md in the Cardprice folder has a PATH-fix block,
# you can paste that here instead.)
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

import pymc as pm
import arviz_base as azb
import arviz_stats as azs
import arviz_plots as azp
print(f"PyMC {pm.__version__}")

try:
    import nutpie
    SAMPLER = "nutpie"
    print("nutpie", nutpie.__version__)
except ImportError:
    SAMPLER = "pymc"

career = eb_data.career(pitcher_rule=3)
alpha0, beta0 = eb_data.load_prior()
print(f"{len(career):,} players | EB prior alpha0 = {alpha0:.1f}, beta0 = {beta0:.1f}")

## 1. Setup and the PyMC 6 / ArviZ 1 API

Most PyMC examples online predate PyMC 6 and ArviZ 1. What changed:
- `pm.sample` returns an xarray **`DataTree`**, not `InferenceData` (`idata.posterior` still works);
- summaries and plots moved: `az.summary` → `arviz_stats.summary`, `az.plot_trace` → `arviz_plots.plot_trace_dist`;
- the default interval is an **89% equal-tailed interval**, not a 94% HDI;
- a dimension can't share a name with a variable.

When in doubt, check the current docs.

## 2. The hierarchical beta-binomial model

Empirical Bayes fixed $\alpha_0, \beta_0$ at their MLE. Full Bayes gives them a prior too. It's easier to put priors on the mean $\mu = \alpha/(\alpha+\beta)$ and the concentration $\kappa = \alpha + \beta$ (that is, $1/\sigma$ from Lesson 06):

$$\mu \sim \text{Beta}(2, 6), \quad \kappa \sim \text{LogNormal}(\log 100, 1.5), \quad p_i \sim \text{Beta}(\mu\kappa, (1-\mu)\kappa), \quad H_i \sim \text{Binomial}(AB_i, p_i).$$

With 10,000 players, sampling every $p_i$ is slow. Instead, **marginalize** them out with `pm.BetaBinomial` (the same likelihood you maximized in Lesson 02), then recover each $p_i$ afterwards from the conjugate update.

**Your turn.** Do a prior predictive check first: sample from the priors on μ and κ, simulate the batting averages they imply, and plot them. Do the priors allow anything absurd? Adjust them if so.

In [ ]:
# Your code here

In [ ]:
# Your code here

## 3. Fit it to all players

**Your turn.** Write the model with coords for players, using `pm.BetaBinomial("H", n=AB, alpha=mu*kappa, beta=(1-mu)*kappa, observed=H)`. Sample with `nuts_sampler=SAMPLER`.

In [ ]:
# Your code here

In [ ]:
# Your code here

**Your turn.** Check the diagnostics (R-hat, ESS, divergences) with `azs.summary`, and plot the traces with `azp.plot_trace_dist`.

In [ ]:
# Your code here

**Your turn.** Convert the posterior draws of μ, κ to α₀ = μκ, β₀ = (1 − μ)κ. Compare their posterior mean and 89% interval with your MLE.

In [ ]:
# Your code here

> **Check:** with ~10,000 players the posterior should be narrow and centred very close to the MLE.

## 4. Compare with empirical Bayes

For each posterior draw of (α₀, β₀), each player's p has a known beta posterior. Draw one p per player per draw: that gives the full-Bayes posterior for every player, including the uncertainty in the prior.

**Your turn.** Compute full-Bayes posterior means and 95% intervals for every player (vectorize over draws; or do it for a random 500 players if memory is tight). Compare them with your EB estimates and intervals: scatter the means, and plot interval width against AB for both.

In [ ]:
# Your code here

In [ ]:
# Your code here

**Reflect.** Do the two approaches differ here? Connect your answer to Lesson 12's results for n = 10,000.

*Your answer:*

## 5. Thirty players: where full Bayes earns its keep

**Your turn.** Repeat Lesson 12's small-sample setting, using known true α₀, β₀ (for example, fitted to all players):
1. sample 30 players' AB, simulate p and H;
2. fit EB (MLE plug-in) and the PyMC model;
3. record each method's 95% coverage of the true p.

Do 20 replications (print progress; each PyMC fit takes a few seconds), then compare the coverage distributions.

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

**Reflect.** Why does full Bayes hold its coverage better when there are only 30 players? What does the posterior for κ look like in those fits compared with the EB point estimate?

*Your answer:*

## 6. Regression in the hierarchy (optional)

**Your turn.** Make μ depend on log(AB) (logit link: `mu = pm.math.invlogit(a + b * log_AB)`) and refit. Compare `b` with your Lesson 06 / 11 estimates. Is its uncertainty what you expected?

In [ ]:
# Your code here

In [ ]:
# Your code here

## 7. Card connection: simulated pull rates

Simulated data keeps this lesson free of data-rights issues (scraped pull rates shouldn't be republished).

**Your turn.** Simulate 40 sets: true pull rate from Beta(3, 60), packs opened from 3 to 300. Fit EB and the PyMC model. For the 10 sets with the fewest packs, plot both 90% intervals and the true rate.

In [ ]:
# Your code here

In [ ]:
# Your code here

**Reflect.** Cardprice notebook 07 uses varying intercepts with a non-centered parameterization for card, set, illustrator and Pokémon effects. How is that model a bigger version of the one in this lesson? What plays the role of κ?

*Your answer:*

## References

- Gelman et al., *Bayesian Data Analysis* (3rd ed.), Section 5.3: the rat tumour example (hierarchical beta-binomial).
- McElreath, *Statistical Rethinking* (2nd ed.), Chapter 12 (beta-binomial) and Chapter 13 (multilevel models).
- PyMC example gallery, "Hierarchical binomial model: rat tumor example": https://www.pymc.io/projects/examples/en/latest/
- PyMC 6 and ArviZ 1 docs: https://www.pymc.io/ and https://python.arviz.org/